# Feature Engineering on High Volume For-Hire Vehicle (HVFHV) Trip Records Dataset:

In this notebook, we are mainly focusing on creating new spark dataframes about the total revenue in different time series for drivers in HVFHV dataset.

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql import functions as F
from pyspark.sql.functions import when, col, count
from pyspark.sql.functions import sum as spark_sum
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_hvfhv_revenue")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"
hvfhv_path = base_dir + '/developed/merged_data/full_hvfhv'
hvfhv_sdf = spark.read.parquet(hvfhv_path)
hvfhv_sdf.show(5)

In [ ]:
hvfhv_sdf = hvfhv_sdf.withColumn(
    "day_type",
    when(hvfhv_sdf["day_of_week"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]), "Weekday")
    .otherwise("Weekend")
)

hvfhv_sdf.show(5)

In [ ]:
num_rows = hvfhv_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hvfhv_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

In [ ]:
hvfhv_sdf.printSchema()

# Hourly Revenue:

Since driver pay is generated after passengers place the order online, it is more accurate to use pickup hour not dropoff hour.

In [ ]:
TOTAL_HOURS = 24

# Group by `pickup_hour` `pickup_date``, `day_type`,
# then sum the driver revenue for each group
hourly_revenue_sdf = hvfhv_sdf.groupBy( "pickup_hour", "pickup_date", "day_type") \
                              .agg(spark_sum("total_revenue").alias("hourly_revenue")) \
                              .orderBy("hourly_revenue")

# Standardize the daily revenue
hourly_revenue_sdf = hourly_revenue_sdf.withColumn('mean_hourly_revenue',
                                                   col('hourly_revenue') / TOTAL_HOURS)

hourly_revenue_sdf = hourly_revenue_sdf.orderBy("pickup_hour")\
                                       .drop("hourly_revenue")

hourly_revenue_sdf.show(5)

In [ ]:
hourly_revenue_sdf.describe().show()

# Hourly Revenue Among Day of Week:

In [ ]:
# Group by `day_of_week`, "day_type", "pickup_date", "pickup_hour" 
# then sum the driver revenue for each group
hourly_revenue_among_day_of_week = hvfhv_sdf.groupBy("pickup_date", "day_of_week",
                                            "day_type", "pickup_hour") \
                                   .agg(spark_sum("total_revenue").alias("day_by_hour_revenue")) \
                                   .orderBy("day_of_week")

# Standardize the hourly revenue
hourly_revenue_among_day_of_week = hourly_revenue_among_day_of_week.withColumn('day_by_hour_revenue',
                                                                               col('day_by_hour_revenue') / TOTAL_HOURS)

hourly_revenue_among_day_of_week.show(5)

In [ ]:
hourly_revenue_among_day_of_week.describe().show()

# Daily Revenue:

In [ ]:
TOTAL_DAYS = 31 + 31 + 30 + 31 + 30 + 31

# Group by `pickup_date` and `PULocationID`, then sum the revenue for each group
daily_revenue_sdf = hvfhv_sdf.groupBy("pickup_date", "PULocationID") \
                             .agg(spark_sum("total_revenue").alias("daily_revenue")) \
                             .orderBy("pickup_date", "PULocationID")

# Standardize the daily revenue
daily_revenue_sdf = daily_revenue_sdf.withColumn('daily_revenue', 
                                                 col('daily_revenue') / TOTAL_DAYS)

daily_revenue_sdf.show(5)

In [ ]:
daily_revenue_sdf.describe().show()

# Monthly Revenue:

Extract monthly revenue in each location ID:

In [ ]:
TOTAL_MONTHS = 6 # the timeline is 6 month

# Group by `month`, 
# then sum the driver revenue for each group
monthly_revenue_sdf = hvfhv_sdf.groupBy("month") \
                               .agg(spark_sum("total_revenue").alias("monthly_revenue")) \
                               .orderBy("month")

# Standardize the monthly revenue
monthly_revenue_sdf = monthly_revenue_sdf.withColumn('monthly_revenue', 
                                                     col('monthly_revenue') / TOTAL_MONTHS)

monthly_revenue_sdf.show(5)

In [ ]:
monthly_revenue_sdf.describe().show()

# Save the Merged Datasets:

Save the hourly revenue dataset:

In [ ]:
hour_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_revenue'
hour_path = os.path.join(hour_dir, file_name)
hourly_revenue_sdf.write.mode('overwrite').parquet(hour_path)

Save the hourly revenue among day of week dataset:

In [ ]:
hourly_revenue_among_day_of_week_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_revenue_among_day_of_week'
hourly_revenue_among_day_of_week_path = os.path.join(hourly_revenue_among_day_of_week_dir, file_name)
hourly_revenue_among_day_of_week.write.mode('overwrite').parquet(hourly_revenue_among_day_of_week_path)

Save the daily revenue dataset:

In [ ]:
day_dir = base_dir + '/developed/merged_data'
file_name = 'daily_revenue'
day_path = os.path.join(day_dir, file_name)
daily_revenue_sdf.write.mode('overwrite').parquet(day_path)

Save the monthly revenue dataset:

In [ ]:
month_dir = base_dir + '/developed/merged_data'
file_name = 'monthly_revenue'
month_path = os.path.join(month_dir, file_name)
monthly_revenue_sdf.write.mode('overwrite').parquet(month_path)